In [ ]:
name = name
sc_id = sc_id # scRNA sc_id: subset from sample
spatial_id = spatial_id 
group = 'celltype_0722'
output = paste0('/data/work/02.result/05.RCTD/rctd_0722/',name)

spatial_path = paste0('/data/work/00.data/spatial_zrc_0526/', spatial_id)
sc_path = paste0("/data/work/00.data/sc/",sc_id)

if (!dir.exists(output)) dir.create(output, recursive = TRUE); setwd(output)

In [ ]:
library(data.table)
library(spacexr)
library(readr)
library(ggplot2)
library(Seurat)
library(tidyverse)

colorlist = c("#FFFF00", "#1CE6FF", "#FF34FF", "#FF4A46", "#008941",
             "#006FA6", "#A30059", "#FFE4E1", "#0000A6", "#63FFAC",
             "#B79762", "#004D43", "#8FB0FF", "#997D87", "#5A0007",
             "#809693", "#1B4400", "#4FC601", "#3B5DFF", "#FF2F80",
             "#BA0900", "#6B7900", "#00C2A0", "#FFAA92", "#FF90C9",
             "#B903AA", "#DDEFFF", "#7B4F4B", "#A1C299", "#0AA6D8",
             "#00A087FF", "#4DBBD5FF", "#E64B35FF", "#3C5488FF", "#F38400",
             "#A1CAF1", "#C2B280", "#848482", "#E68FAC", "#0067A5",
             "#F99379", "#604E97", "#F6A600", "#B3446C", "#DCD300",
             "#882D17", "#8DB600", "#654522", "#E25822", "#2B3D26",
             "#191970", "#000080","#6495ED", "#1E90FF", "#00BFFF", 
             "#00FFFF", "#FF1493","#FF00FF", "#A020F0", "#63B8FF", 
             "#008B8B", "#54FF9F","#00FF00", "#76EE00", "#FFF68F")

## 导入空间组数据
SeuObj = readRDS(spatial_path)
print("spatial read successfully")

## 导入单细胞数据
sc = readRDS(sc_path)
kc_meta <- read.csv('/data/work/02.result/05.RCTD/rctd_0722/kc_metadata_0722.csv',row.names = 1)
sc$celltype_0722 <- sc$celltype_0611
sc@meta.data[rownames(kc_meta),'celltype_0722'] <- kc_meta$kc_celltype
table(sc$celltype_0722)
# meta = read.csv(metadata,row.names=1)
# sc@meta.data = meta[colnames(sc),]
# sc = subset(sc,sample == sc_id)

# 设置分组变量
Idents(sc) <- sc@meta.data[[group]]

# 1. 下采样并确保无 NA
set.seed(123)
sc_downsampled <- subset(sc, downsample = 500)
sc_downsampled <- sc_downsampled[, !is.na(sc_downsampled@meta.data[[group]])]
sc_downsampled@meta.data[[group]] <- as.factor(sc_downsampled@meta.data[[group]])

# 2. 移除小于 25 个细胞的群体
cell_low <- table(sc_downsampled@meta.data[[group]])[table(sc_downsampled@meta.data[[group]]) < 25] %>% names()
if (length(cell_low) > 0) {
  message("Removing small clusters after downsampling: ", paste(cell_low, collapse = ", "))
  sc_downsampled <- subset(sc_downsampled, idents = cell_low, invert = TRUE)
  sc_downsampled@meta.data[[group]] <- droplevels(sc_downsampled@meta.data[[group]])
} else {
  message("No clusters below 25 cells.\n")
}

# 3. 打印最终状态
print(table(sc_downsampled@meta.data[[group]]))
num_na <- sum(is.na(sc_downsampled@meta.data[[group]])) # 统计 NA 的数量
print(paste("Number of NA values:", num_na))
print("sc read successfully！")


## run RCTD

###提取空间组数据的信息
exp_spatial=as.matrix(SeuObj@assays$Spatial@counts)
exp_spatial=as.data.frame(exp_spatial)

SeuObj$coor_y = SeuObj@images[[names(SeuObj@images)]]@coordinates$row
SeuObj$coor_x = SeuObj@images[[names(SeuObj@images)]]@coordinates$col
coord <- data.frame(stereo_1 = SeuObj$coor_x, stereo_2 = SeuObj$coor_y)
SeuObj[["stereo"]] <- SeuratObject::CreateDimReducObject(embeddings = as.matrix(coord), key = "stereo_", assay = "SCT")

row <- SeuObj$coor_y
col <- SeuObj$coor_x
coord_spatial <- data.frame(row,col)
rownames(coord_spatial)<- Cells(SeuObj)
nUMI_spatial=SeuObj@meta.data[,"nCount_Spatial"]
names(nUMI_spatial)=rownames(SeuObj@meta.data)

# ------------------------------------------------------
# 提取下采样后的单细胞数据
exp_sc = as.matrix(sc_downsampled@assays$RNA@counts)
exp_sc = as.data.frame(exp_sc)

celltype_sc = sc_downsampled@meta.data[[group]]
names(celltype_sc) = rownames(sc_downsampled@meta.data)
celltype_sc = as.factor(celltype_sc)

nUMI_sc = sc_downsampled@meta.data[,"nCount_RNA"]
names(nUMI_sc) = rownames(sc_downsampled@meta.data)


###构建RCTD的数据数据（puck和reference）
reference <- Reference(exp_sc, celltype_sc, nUMI_sc)
puck <- SpatialRNA(coord_spatial, exp_spatial, nUMI_spatial)
print("RCTD data have praperad successfully！")

###清除内存
rm(sc)
gc()

In [ ]:
myRCTD <- create.RCTD(puck, reference, max_cores = 10, UMI_min=0, CELL_MIN_INSTANCE = 25, UMI_min_sigma = 200)
myRCTD <- run.RCTD(myRCTD, doublet_mode = 'doublet' )

saveRDS(myRCTD, file = paste0(name, "_myRCTD.rds"))
print("RCTD successfully！")

## 查看映射结果
head(myRCTD@results$results_df)
df = myRCTD@results$results_df
df$first_type <- as.character(df$first_type)
df$first_type[df$spot_class == 'reject'] <- "unknown"
table(df$first_type, useNA = 'always')  # 查看含 NA 情况

## 添加 meta 信息
rownames(df) <- rownames(myRCTD@results$results_df) # 设置 rownames（即 spot ID）
named_vector <- setNames(df$first_type, rownames(df)) # 构建命名向量
head(named_vector)

## 添加 meta 信息
SeuObj$RCTD <- NA_character_ # 构造与 SeuObj 维度一致的 vector，默认填 NA 或 'unknown'
SeuObj$RCTD[colnames(SeuObj)] <- named_vector[colnames(SeuObj)]
SeuObj$RCTD[is.na(SeuObj$RCTD)] <- "unknown"
table(SeuObj$RCTD, useNA = "always")
write.csv(SeuObj@meta.data, file = paste0(name, '.RCTD.metadata.csv'), row.names = TRUE)


## 空间可视化
###可视化映射得分
df=myRCTD@results$weights
df=as.data.frame(as.matrix(df))
df$x=myRCTD@spatialRNA@coords$x
df$y=myRCTD@spatialRNA@coords$y
df$p_size <- rep(1000,times=nrow(df))
                    
pdf(paste0(sc_id,"-weight.pdf"),height=10,width=10,bg = "black")
apply(as.matrix(colnames(df)[1:(ncol(df)-3)]),1,function(xx){
  ggplot(df,aes(x=x,y=y,fill=df[,xx]))+
        geom_tile()+coord_fixed()+ 
        scale_fill_gradientn(colours = c("#3E4A89FF", "#B4DE2CFF", "#FF0000"))+ #"#FDE725FF"
        labs(title=xx)+theme_void()+
        theme(panel.background = element_rect(fill = "black"),
              legend.title = element_blank(),
              plot.background = element_rect(fill = "black"),
              text = element_text(color = "white"),
              plot.title = element_text(color = "white"))
})
dev.off()
                    
###可视化映射celltype分布
results_df<- as.data.frame(myRCTD@results[["results_df"]])
results_df<- results_df %>% mutate("cell_type" = ifelse(spot_class=="reject","unknown",first_type))
#results_df[results_df$cell_type != "unknow",]$cell_type=as.vector(results_df[results_df$cell_type != "unknow",]$first_type) 

results_df$x=myRCTD@spatialRNA@coords$x
results_df$y=myRCTD@spatialRNA@coords$y
cell_name <- unique(results_df$first_type)
cell_color<- colorlist[1:length(cell_name)]

p1 <- ggplot(results_df,aes(x=x,y=y,fill=first_type))+
      geom_tile()+coord_fixed()+
      labs(title="all_cell")+theme_void()+
      theme(panel.background = element_rect(fill = "black"),
            plot.background = element_rect(fill = "black"),
            text = element_text(color = "white"),
            plot.title = element_text(color = "white"))+
      theme(legend.position="top",legend.key.size=unit(1,'mm'),legend.key.width=unit(1,'mm'))+
      scale_fill_manual(values = cell_color)
#dev.off()
ggsave(paste0(sc_id, "-cell_type.pdf"),plot=p1,height = 10,width = 10)

## 查看RCTD结果
results = myRCTD@results
norm_weights = sweep(results$weights,1,rowSums(results$weights),'/')
cell_type_names = myRCTD@cell_type_info$info[[2]]
spatialRNA <- myRCTD@spatialRNA
resultsdir <- paste0(sc_id, '_RCTD_Plots') ## you may change this to a more accessible directory on your computer.
dir.create(resultsdir)
plot_weights(cell_type_names, spatialRNA, resultsdir, norm_weights) ### 绘制full_mode模式下每种细胞类型的可信权重 (saved as 'results/cell_type_weights_unthreshold.pdf')
plot_weights_unthreshold(cell_type_names, spatialRNA, resultsdir, norm_weights) #### 绘制full_mode模式下每种细胞类型的权重; 这里每种细胞类型一幅图，点表示空间上的一个像素或者spot，颜色为权重 (saved as 'results/cell_type_weights.pdf')
plot_weights_doublet(cell_type_names, spatialRNA, resultsdir, results$weights_doublet, results$results_df) ### 绘制doublet_mode模式下每种细胞类型的权重 (saved as 'results/cell_type_weights_doublets.pdf')